In [ ]:
# ============================================
# 0. 环境准备
# ============================================
# 安装必要的库（如果已安装可注释掉）
# !pip install pyspark findspark xgboost sparkxgb synapseml

import findspark
findspark.init()

import os
import shutil
import time
import pandas as pd
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.functions import (
    col, window, to_timestamp, from_unixtime, regexp_extract
)

# 创建 SparkSession
spark = SparkSession.builder \
    .appName("Assignment3") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.streaming.checkpointLocation", "/tmp/spark_ckpt") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session 已启动")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 16:55:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# ============================================
# Q1: IoT 设备流式分析
# ============================================

# ---- 路径配置 ----
Q1_SRC = "Q1_IOT/iot_devices.json"
Q1_SPLIT_DIR = "Q1_IOT/split_input"
Q1_OUT1 = "Q1_IOT/output/device_type_count"
Q1_OUT2 = "Q1_IOT/output/low_battery_count"
CKPT1_1 = "Q1_IOT/checkpoint/task1"
CKPT1_2 = "Q1_IOT/checkpoint/task2"

# 清理旧输出（避免重复运行冲突）
for p in [Q1_SPLIT_DIR, Q1_OUT1, Q1_OUT2, CKPT1_1, CKPT1_2]:
    try:
        shutil.rmtree(p)
    except:
        pass

# ---- 拆分原始 JSON 以模拟流式输入 ----
static_df = spark.read.json(Q1_SRC)
print(f"原始数据行数: {static_df.count()}")
static_df.printSchema()

# 重分区为 10 个小文件
static_df.repartition(10).write.mode("overwrite").json(Q1_SPLIT_DIR)
print(f"拆分完成，文件数: {len(os.listdir(Q1_SPLIT_DIR))}")

# ---- 定义 Schema 并创建流式 DataFrame ----
schema = T.StructType([
    T.StructField("device_name", T.StringType(), True),
    T.StructField("cn", T.StringType(), True),
    T.StructField("battery_level", T.IntegerType(), True),
    T.StructField("timestamp", T.DoubleType(), True)   # 毫秒时间戳
])

stream_df = spark.readStream \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .json(Q1_SPLIT_DIR)

# 添加事件时间并设置 watermark（允许迟到10秒）
df_with_time = stream_df \
    .withColumn("event_time", to_timestamp(from_unixtime(col("timestamp") / 1000))) \
    .withWatermark("event_time", "10 seconds")

# ---- 任务 (1) 提取设备类型并按类型+国家计数 ----
df_typed = df_with_time.withColumn(
    "device_type",
    regexp_extract(col("device_name"), r'^([^-]+-[^-]+)', 1)
)

result1 = df_typed.groupBy(
    window(col("event_time"), "1 second"),   # 1秒滚动窗口，无重叠
    col("device_type"),
    col("cn")
).count().select(
    col("window"), col("device_type"), col("cn"), col("count")
).orderBy(col("count").desc())

query1 = result1.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", Q1_OUT1) \
    .option("checkpointLocation", CKPT1_1) \
    .start()

# ---- 任务 (2) 低电量设备（<=5）按国家计数 ----
low_battery = df_with_time \
    .filter(col("battery_level") <= 5) \
    .groupBy(
        window(col("event_time"), "1 second"),
        col("cn")
    ).count().select(
        col("window"), col("cn"), col("count")
    )

query2 = low_battery.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", Q1_OUT2) \
    .option("checkpointLocation", CKPT1_2) \
    .start()

# 等待流处理完成（根据数据量调整）
query1.awaitTermination(30)
query2.awaitTermination(30)
print("Q1 流处理完成")

NameError: name 'user_schema' is not defined

In [ ]:
# ============================================
# Q2: 用户活动数据滑动窗口聚合
# ============================================

# ---- 路径配置 ----
Q2_INPUT = "Q2_activity_data"
Q2_PARQUET_OUT = "Q2_activity_data/output_parquet"
CKPT2_MEM = "Q2_activity_data/checkpoint/mem"
CKPT2_PARQUET = "Q2_activity_data/checkpoint/parquet"

for p in [Q2_PARQUET_OUT, CKPT2_MEM, CKPT2_PARQUET]:
    try:
        shutil.rmtree(p)
    except:
        pass

# ---- 流式读取 JSON 文件 ----
activity_stream = spark.readStream \
    .option("maxFilesPerTrigger", 1) \
    .json(Q2_INPUT) \
    .repartition(3)   # 设置分区数为 3

# 提取事件时间（假设 Creation_Time 为毫秒时间戳）
df_activity = activity_stream \
    .withColumn("event_time", to_timestamp(from_unixtime(col("Creation_Time") / 1000))) \
    .withWatermark("event_time", "10 minutes")

# ---- 滑动窗口聚合（6分钟窗口，步长3分钟） ----
sliding_counts = df_activity.groupBy(
    window(col("event_time"), "6 minutes", "3 minutes"),
    col("user")
).count()

# 任务 (1) 写入内存 + update 模式
query_mem = sliding_counts.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("activity_query") \
    .option("checkpointLocation", CKPT2_MEM) \
    .start()

# 等待部分数据到达后显示前3条结果
time.sleep(60)  # 等待一分钟，可根据实际情况调整
print("内存表 activity_query 的前3条记录：")
spark.sql("SELECT * FROM activity_query LIMIT 3").show(truncate=False)

# 任务 (2) 同时写入 Parquet（append 模式）
query_parquet = sliding_counts.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", Q2_PARQUET_OUT) \
    .option("checkpointLocation", CKPT2_PARQUET) \
    .start()

# 继续等待直到所有数据写完
query_mem.awaitTermination(60)
query_parquet.awaitTermination(60)
print("Q2 流处理完成")

In [ ]:
# ============================================
# Q3: 房价预测（决策树 / XGBoost / LightGBM）
# ============================================

# ---- 路径配置 ----
TRAIN_PATH = "Q3_house_price_data/train.csv"
TEST_PATH = "Q3_house_price_data/test.csv"
OUTPUT_CSV = "Q3_house_price_data/predictions.csv"

# ---- 读取数据 ----
train = spark.read.csv(TRAIN_PATH, header=True, inferSchema=True)
test = spark.read.csv(TEST_PATH, header=True, inferSchema=True)

target = "SalePrice"
id_col = "Id"

# ---- 识别类别列与数值列 ----
# 根据数据描述文档，将 MSSubClass 等实际为类别的数值列转换为字符串
categorical_cols = [
    "MSSubClass", "MSZoning", "Street", "Alley", "LotShape", "LandContour",
    "Utilities", "LotConfig", "LandSlope", "Neighborhood", "Condition1",
    "Condition2", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl",
    "Exterior1st", "Exterior2nd", "MasVnrType", "ExterQual", "ExterCond",
    "Foundation", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
    "BsmtFinType2", "Heating", "HeatingQC", "CentralAir", "Electrical",
    "KitchenQual", "Functional", "FireplaceQu", "GarageType", "GarageFinish",
    "GarageQual", "GarageCond", "PavedDrive", "PoolQC", "Fence",
    "MiscFeature", "SaleType", "SaleCondition"
]

for c in categorical_cols:
    if c in train.columns:
        train = train.withColumn(c, col(c).cast("string"))
        test = test.withColumn(c, col(c).cast("string"))

# 数值列（排除目标、ID 和类别列）
numeric_cols = [
    c for c in train.columns
    if c not in categorical_cols + [target, id_col]
    and train.schema[c].dataType.typeName() in ('double','float','int','long','short','integer')
]

print(f"类别列: {len(categorical_cols)} 个, 数值列: {len(numeric_cols)} 个")

# ---- 特征工程 Pipeline ----
from pyspark.ml.feature import StringIndexer, VectorAssembler, Imputer
from pyspark.ml import Pipeline

# 类别列索引化
indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    for c in categorical_cols if c in train.columns
]

# 数值列缺失值填充（中位数）
imputer = Imputer(inputCols=numeric_cols, outputCols=numeric_cols, strategy="median")

# 特征向量
feature_cols = [c+"_idx" for c in categorical_cols if c in train.columns] + numeric_cols
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")

pipeline = Pipeline(stages=[imputer] + indexers + [assembler])

# ---- 划分训练集 / 验证集 ----
train_data, val_data = train.randomSplit([0.8, 0.2], seed=42)

# 拟合 pipeline 并转换所有数据集
pipeline_model = pipeline.fit(train_data)
train_processed = pipeline_model.transform(train_data)
val_processed = pipeline_model.transform(val_data)
test_processed = pipeline_model.transform(test)

train_processed.select("features", target).show(5, truncate=False)

# ---- 模型评估函数 ----
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="rmse")

# ======================
# 模型1: 决策树
# ======================
print("===== 训练决策树 =====")
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

dt = DecisionTreeRegressor(featuresCol="features", labelCol=target, seed=42)
dt_paramGrid = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [5, 10, 15]) \
    .addGrid(dt.maxBins, [32, 64]) \
    .build()

cv_dt = CrossValidator(estimator=dt, estimatorParamMaps=dt_paramGrid,
                       evaluator=evaluator, numFolds=3, seed=42)
cv_model_dt = cv_dt.fit(train_processed)

val_pred_dt = cv_model_dt.transform(val_processed)
rmse_dt = evaluator.evaluate(val_pred_dt)
print(f"决策树验证集 RMSE: {rmse_dt:.4f}")

test_pred_dt = cv_model_dt.transform(test_processed)

# ======================
# 模型2: XGBoost
# ======================
print("===== 训练 XGBoost =====")
from sparkxgb import XGBoostRegressor

xgb = XGBoostRegressor(
    featuresCol="features", labelCol=target,
    missing=0.0, n_estimators=200, max_depth=6,
    learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
    seed=42
)
model_xgb = xgb.fit(train_processed)

val_pred_xgb = model_xgb.transform(val_processed)
rmse_xgb = evaluator.evaluate(val_pred_xgb)
print(f"XGBoost 验证集 RMSE: {rmse_xgb:.4f}")

test_pred_xgb = model_xgb.transform(test_processed)

# ======================
# 模型3: LightGBM
# ======================
print("===== 训练 LightGBM =====")
from synapse.ml.lightgbm import LightGBMRegressor

lgb = LightGBMRegressor(
    featuresCol="features", labelCol=target,
    numLeaves=63, learningRate=0.05, numIterations=200,
    baggingFraction=0.8, baggingFreq=1, featureFraction=0.8,
    seed=42
)
model_lgb = lgb.fit(train_processed)

val_pred_lgb = model_lgb.transform(val_processed)
rmse_lgb = evaluator.evaluate(val_pred_lgb)
print(f"LightGBM 验证集 RMSE: {rmse_lgb:.4f}")

test_pred_lgb = model_lgb.transform(test_processed)

# ---- 选择最佳模型输出预测文件 ----
# 根据验证集 RMSE，这里以 LightGBM 为例（请根据实际运行结果修改）
print("===== 输出预测文件 =====")
best_pred = test_pred_lgb.select(col(id_col), col("prediction").alias(target))

# 转为 Pandas 输出单个 CSV
pd_pred = best_pred.toPandas()
pd_pred.to_csv(OUTPUT_CSV, index=False)
print(f"预测文件已保存至: {OUTPUT_CSV}")
print("前5条预测结果：")
print(pd_pred.head())